# Merge municipio voting data with manifesto scores and party families (2022, 2026)

Inputs:
- `data/electoral/processed/voting_data_2022_2026.xlsx` (sheet `Voting Data`): one row per municipio x candidate, votes in wide form (`NUM_VOT_2022`, `NUM_VOT_2026`), candidate in `DES_CAN`.
- `data/manifestos/output/policlim_comparison/manifesto_scores_families_2022_2026.xlsx` (sheet `scores`): one row per manifesto (candidate x year) with the five scores and the `pro_mining` / `reformist` / `contra_mining` dummies. Claudia López is already excluded there.

The scores file identifies each manifesto by `candidate` + `year` (its `candidate` column was built to match `DES_CAN` spelling, and every candidate has exactly one manifesto per year), so the votes are first reshaped to long form (`year`, `candidate`) and then merged on that key. `manifesto_label` is carried along.

Output: `data/electoral/processed/voting_manifesto_merged_2022_2026.xlsx`, one row per municipio x year x candidate.

In [3]:
from pathlib import Path

import pandas as pd

# This notebook lives in code/analysis/, so the project root is two levels up
# (assumes the kernel's working directory is the notebook's folder, the Jupyter default).
PROJECT_ROOT = Path.cwd().resolve().parents[1]

VOTES_PATH = PROJECT_ROOT / "data" / "electoral" / "processed" / "voting_data_2022_2026.xlsx"
SCORES_PATH = PROJECT_ROOT / "data" / "manifestos" / "output" / "policlim_comparison" / "manifesto_scores_families_2022_2026.xlsx"
OUTPUT_PATH = PROJECT_ROOT / "data" / "electoral" / "processed" / "voting_manifesto_merged_2022_2026.xlsx"

for path in (VOTES_PATH, SCORES_PATH):
    assert path.exists(), f"Input not found: {path}"

In [4]:
votes_wide = pd.read_excel(VOTES_PATH, sheet_name="Voting Data")
# The "scores" sheet already contains the pro_mining / reformist / contra_mining dummies,
# so the separate "party_families" sheet is not needed here.
scores = pd.read_excel(SCORES_PATH, sheet_name="scores")

print(f"votes (wide): {votes_wide.shape}, scores: {scores.shape}")
print("vote columns:", list(votes_wide.columns))
print("score columns:", list(scores.columns))

votes (wide): (26804, 7), scores: (8, 14)
vote columns: ['COD_DDE', 'DES_DD', 'COD_MME', 'DES_MM', 'DES_CAN', 'NUM_VOT_2026', 'NUM_VOT_2022']
score columns: ['year', 'candidate', 'party', 'manifesto_label', 'climate_salience_share', 'pro_just_transition', 'anti_ff', 'pro_ff', 'energy_security', 'sentence_count', 'pro_mining', 'reformist', 'contra_mining', 'stance']


In [5]:
# Wide (NUM_VOT_2022, NUM_VOT_2026) -> long (year, votes): one row per municipio x year x candidate.
# A missing vote count means the candidate did not run in that year, so those rows are dropped.
MUNICIPIO_COLUMNS = ["COD_DDE", "DES_DD", "COD_MME", "DES_MM"]

votes = (
    votes_wide
    .melt(
        id_vars=MUNICIPIO_COLUMNS + ["DES_CAN"],
        value_vars=["NUM_VOT_2022", "NUM_VOT_2026"],
        var_name="year",
        value_name="votes",
    )
    .dropna(subset=["votes"])
    .rename(columns={"DES_CAN": "candidate"})
)
votes["year"] = votes["year"].str.extract(r"(\d{4})")[0].astype(int)
votes["votes"] = votes["votes"].astype(int)

# Vote share within each municipio-year. The denominator is all rows of that municipio-year,
# i.e. it includes blank, null and unmarked ballots (Votos En Blanco / Nulos / No Marcados).
votes["vote_share"] = votes["votes"] / votes.groupby(MUNICIPIO_COLUMNS[::2] + ["year"])["votes"].transform("sum")

print(f"votes (long): {votes.shape}")
votes.head()

votes (long): (31427, 8)


,COD_DDE,DES_DD,COD_MME,DES_MM,candidate,year,votes,vote_share
3,1,ANTIOQUIA,1,MEDELLIN,Enrique Gómez Martínez,2022,2568,0.002256
4,1,ANTIOQUIA,1,MEDELLIN,Federico Gutiérrez,2022,603374,0.530067
6,1,ANTIOQUIA,1,MEDELLIN,Gustavo Petro,2022,275780,0.242274
7,1,ANTIOQUIA,1,MEDELLIN,Ingrid Betancourt,2022,572,0.000503
9,1,ANTIOQUIA,1,MEDELLIN,John Milton Rodríguez,2022,7770,0.006826


In [6]:
# One manifesto per candidate-year, so (year, candidate) is the manifesto key. Left merge keeps
# candidates without a manifesto (minor candidates, blank/null ballots) with NaN scores.
merged = votes.merge(scores, on=["year", "candidate"], how="left", validate="many_to_one")
merged["has_manifesto"] = merged["manifesto_label"].notna()

print(f"{len(merged):,} rows, {int(merged['has_manifesto'].sum()):,} with manifesto scores")
merged.head()

31,427 rows, 9,506 with manifesto scores


,COD_DDE,DES_DD,COD_MME,DES_MM,candidate,year,votes,vote_share,party,manifesto_label,...,pro_just_transition,anti_ff,pro_ff,energy_security,sentence_count,pro_mining,reformist,contra_mining,stance,has_manifesto
0,1,ANTIOQUIA,1,MEDELLIN,Enrique Gómez Martínez,2022,2568,0.002256,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1,ANTIOQUIA,1,MEDELLIN,Federico Gutiérrez,2022,603374,0.530067,Equipo por Colombia,Gutiérrez (Equipo por Colombia),...,0.003597,0.006295,0.023381,0.012590,1112.0,1.0,0.0,0.0,Pro-mining,True
2,1,ANTIOQUIA,1,MEDELLIN,Gustavo Petro,2022,275780,0.242274,Pacto Histórico,Petro (Pacto Histórico),...,0.043536,0.047493,0.003958,0.010554,758.0,0.0,0.0,1.0,Contra-mining,True
3,1,ANTIOQUIA,1,MEDELLIN,Ingrid Betancourt,2022,572,0.000503,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1,ANTIOQUIA,1,MEDELLIN,John Milton Rodríguez,2022,7770,0.006826,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


In [7]:
# Match diagnostics: who on each side found no partner.
print("Candidates / ballot categories without a manifesto score, by year (kept with NaN scores):")
print(merged.loc[~merged["has_manifesto"]].groupby("year")["candidate"].unique().to_string())
print()

scores_matched = scores.merge(
    votes[["year", "candidate"]].drop_duplicates(), on=["year", "candidate"], how="left", indicator=True
)
unmatched_scores = scores_matched.loc[scores_matched["_merge"] == "left_only", ["year", "candidate", "manifesto_label"]]
print("Manifestos with no voting rows (should be empty):")
print(unmatched_scores.to_string(index=False) if not unmatched_scores.empty else "  none")

Candidates / ballot categories without a manifesto score, by year (kept with NaN scores):
year
2022    [Enrique Gómez Martínez, Ingrid Betancourt, Jo...
2026    [Carlos Eduardo Caicedo Omar, Claudia López, G...

Manifestos with no voting rows (should be empty):
  none


In [8]:
merged.to_excel(OUTPUT_PATH, index=False, sheet_name="merged")
print(f"Saved {len(merged):,} rows: {OUTPUT_PATH}")

Saved 31,427 rows: C:\Users\charlott\Dropbox (Personal)\Green_Backlash_Colombia\data\electoral\processed\voting_manifesto_merged_2022_2026.xlsx
